In [ ]:
import numpy as np

import pandas as pd

In [ ]:
# download at OneDrive: https://1drv.ms/f/c/940BFF2A3D86BFF0/IgDovkTv0NjsRZWpGpK_EBL2AUIna1CmxEd5dcoM2_Hn57A?e=unANc7
df = pd.read_csv("/content/campaigns.csv")

In [4]:
# Hiển thị các thông tin cơ bản
print(f"Số dòng: {len(df)}")
print(f"Số cột: {len(df.columns)}")
df.head()

Số dòng: 166665
Số cột: 29


,Campaign_ID,Brand,Campaign_Type,Target_Audience,Channel_Used,Primary_Channel,Channel_Count,Duration,Duration_Group,Language,...,Revenue_Per_Conversion,CPA,Total_Spend,Profit,CTR,Lead_Rate,Conversion_Rate,ROI,Engagement_Score,ROI_Category
0,NY-CMP-13007,Nykaa,Paid Ads,Youth,"Email, YouTube, Facebook",Email,3,29,Very Long (22+ days),Hindi,...,685,9.08,60708.88,4519201.12,0.1470,0.5956,0.7678,74.42,30.17,Excellent
1,NY-CMP-22689,Nykaa,Influencer,Youth,"Instagram, YouTube",Instagram,2,20,Long (15-21 days),Hindi,...,719,10.68,56764.20,3764720.80,0.1409,0.5464,0.7095,66.32,27.26,Excellent
2,NY-CMP-49715,Nykaa,Paid Ads,Premium Shoppers,WhatsApp,WhatsApp,1,12,Medium (8-14 days),Tamil,...,797,12.08,56341.12,3660866.88,0.1275,0.5946,0.6881,65.00,25.54,Excellent
3,NY-CMP-26356,Nykaa,Paid Ads,Premium Shoppers,Facebook,Facebook,1,5,Short (1-7 days),Hindi,...,706,10.84,51641.76,3311742.24,0.1465,0.4891,0.7316,64.14,27.06,Excellent
4,NY-CMP-39329,Nykaa,Paid Ads,Youth,"Google, Facebook, YouTube",Google,3,27,Very Long (22+ days),Hindi,...,728,12.01,52267.52,3115988.48,0.1443,0.5271,0.7376,59.64,27.64,Excellent


In [5]:
# Kiểm tra thông tin các cột
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 166665 entries, 0 to 166664
Data columns (total 29 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   Campaign_ID             166665 non-null  object 
 1   Brand                   166665 non-null  object 
 2   Campaign_Type           166665 non-null  object 
 3   Target_Audience         166665 non-null  object 
 4   Channel_Used            166665 non-null  object 
 5   Primary_Channel         166665 non-null  object 
 6   Channel_Count           166665 non-null  int64  
 7   Duration                166665 non-null  int64  
 8   Duration_Group          166665 non-null  object 
 9   Language                166665 non-null  object 
 10  Customer_Segment        166665 non-null  object 
 11  Campaign_Date           166665 non-null  object 
 12  Campaign_Month          166665 non-null  object 
 13  Campaign_Quarter        166665 non-null  object 
 14  Impressions         

In [6]:
# Kiểm tra số nhóm trước khi xử lý whitespace
print("Số nhóm trước khi làm sạch:")
print(df.groupby('Brand')['Revenue'].sum())

# FIX: loại bỏ khoảng trắng đầu/cuối
df['Brand'] = df['Brand'].str.strip()

# Group lại sau khi làm sạch
brand_revenue = (
    df.groupby('Brand')['Revenue']
      .sum()
      .sort_values(ascending=False)
)

print("\nSố nhóm sau khi làm sạch:")
print(brand_revenue)

Số nhóm trước khi làm sạch:
Brand
Nykaa          28656364282
Purplle        28540346627
Tira Beauty    28453535162
Name: Revenue, dtype: int64

Số nhóm sau khi làm sạch:
Brand
Nykaa          28656364282
Purplle        28540346627
Tira Beauty    28453535162
Name: Revenue, dtype: int64


In [8]:
# Simulate: inject khoảng trắng vào 1/7 và 1/11 dòng
df_dirty = df.copy()
mask1 = df_dirty.index % 7 == 0
mask2 = df_dirty.index % 11 == 0
df_dirty.loc[mask1, 'Campaign_Type'] += ' '
df_dirty.loc[mask2, 'Campaign_Type'] = ' ' + df_dirty.loc[mask2, 'Campaign_Type']

# ── Code A: groupby KHÔNG xử lý ──────────────────────
result_a  = df_dirty.groupby('Campaign_Type')['Revenue'].sum()
unique_a  = df_dirty['Campaign_Type'].nunique()

# ── Code B: strip() rồi groupby ──────────────────────
df_clean = df_dirty.copy()
df_clean['Campaign_Type'] = df_clean['Campaign_Type'].str.strip()
result_b = df_clean.groupby('Campaign_Type')['Revenue'].sum()
unique_b = df_clean['Campaign_Type'].nunique()

print(f"Code A – số nhóm Campaign_Type: {unique_a}")
print(f"Code B – số nhóm Campaign_Type: {unique_b}")

Code A – số nhóm Campaign_Type: 20
Code B – số nhóm Campaign_Type: 5


In [13]:
if 'CTR' not in df.columns:
    df['CTR'] = df['Clicks'] / df['Impressions']

# Group theo Campaign_Type
result = (
    df.groupby('Campaign_Type')
      .agg(
          Total_Impressions=('Impressions', 'sum'),
          Total_Clicks=('Clicks', 'sum'),
          Avg_CTR=('CTR', 'mean')
      )
)

# Weighted Average CTR
result['Weighted_CTR'] = result['Total_Clicks'] / result['Total_Impressions']
# Đổi sang %
result['Avg_CTR'] = (result['Avg_CTR'] * 100).round(2)
result['Weighted_CTR'] = (result['Weighted_CTR'] * 100).round(2)

print(result.reset_index())

# Trung bình cộng CTR giữa các nhóm
simple_avg = result['Avg_CTR'].mean()

# Weighted CTR toàn bộ dataset
weighted_avg = (
    result['Total_Clicks'].sum()
    / result['Total_Impressions'].sum()
) * 100

print(f"Simple AVG CTR:   {simple_avg:.2f}%")
print(f"Weighted AVG CTR: {weighted_avg:.2f}%")

  Campaign_Type  Total_Impressions  Total_Clicks  Avg_CTR  Weighted_CTR
0         Email         1840785851     156471621     8.48          8.50
1    Influencer         1837177162     156531226     8.53          8.52
2      Paid Ads         1838406238     156264327     8.51          8.50
3           SEO         1828167042     154817943     8.47          8.47
4  Social Media         1832180954     156301670     8.53          8.53
Simple AVG CTR:   8.50%
Weighted AVG CTR: 8.50%
